# 6단계 — Pastry/Other_Faults 보조 분류기 추가

**목표:** 5단계 Bumps/Other_Faults 보조 모델을 고정한 상태에서, Pastry ↔ Other_Faults만 추가로 재판정하는 2번째 보조 모델이 Macro F1을 높이는지 확인합니다.

- 비교 모델: (1) 기존 2모델 스태킹, (2) 기존 + Bumps/Other_Faults 보조, (3) 기존 + Pastry/Other_Faults 보조, (4) 두 보조 모델 순차 적용
- 모델 학습은 각 **외부 학습 Fold만** 사용하므로 외부 검증 Fold의 정답이 학습에 들어가지 않습니다.
- B/O 설정은 5단계 고정값(confidence=0.58, gap≤0.30), P/O 설정은 이전 탐색에서 제안된 보수적 후보(confidence=0.65, gap≤0.15)로 **시드 2053 재검증 전에 고정**했습니다.
- 기존 2027/2041 분할은 이미 탐색 또는 검토에 사용했으므로, 새로운 검증 분할 `random_state=2053`를 사용합니다.
- 새 분할 역시 동일한 1,552행 개발 데이터를 재분할한 것이므로 **독립 테스트셋 평가로 간주하면 안 됩니다**. 389행 홀드아웃은 이번 실험에서 건드리지 않습니다.
- 목표 0.835는 **외부 Fold별 Macro F1 평균**으로 평가합니다.

**실행:** `Faults.NNA`와 `Faults27x7_var`를 같은 폴더에 두고 셀을 위에서부터 실행합니다. 1~5단계 노트북 실행 불필요.

In [1]:
# 필수 패키지: 미설치 시 아래 주석 해제 후 실행
# %pip install lightgbm scikit-learn pandas numpy

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from lightgbm import LGBMClassifier

warnings.filterwarnings('ignore', category=FutureWarning)
root = Path('.')
for fname in ['Faults.NNA', 'Faults27x7_var']:
    if not (root/fname).exists():
        raise FileNotFoundError(f'{fname} 파일을 이 노트북과 같은 폴더에 넣어 주세요.')
cols = [line.strip() for line in (root/'Faults27x7_var').read_text(encoding='utf-8').splitlines() if line.strip()]
raw = pd.read_csv(root/'Faults.NNA', sep=r'\s+', header=None, names=cols)
assert raw.shape == (1941, 34), f'데이터 형식 확인 필요: {raw.shape}'
encoder = LabelEncoder()
y_full = encoder.fit_transform(raw[cols[27:]].idxmax(axis=1))
X_full = raw.iloc[:, :27].drop(columns='TypeOfSteel_A300')
idx_dev, idx_test = train_test_split(
    np.arange(len(y_full)), test_size=0.2, stratify=y_full, random_state=42
)
X_dev = X_full.iloc[idx_dev].reset_index(drop=True)
y_dev = y_full[idx_dev]
B = int(encoder.transform(['Bumps'])[0])
O = int(encoder.transform(['Other_Faults'])[0])
print('개발 데이터:', X_dev.shape, '| 사용하지 않는 홀드아웃:', len(idx_test))
print('클래스:', list(encoder.classes_))

개발 데이터: (1552, 26) | 사용하지 않는 홀드아웃: 389
클래스: ['Bumps', 'Dirtiness', 'K_Scatch', 'Other_Faults', 'Pastry', 'Stains', 'Z_Scratch']


In [2]:
# 기존 3단계와 동일한 ET/LGB 스태킹 설정
et_base = ExtraTreesClassifier(
    random_state=42, n_jobs=1, n_estimators=500,
    min_samples_split=5, min_samples_leaf=1,
    max_features=0.8, max_depth=None,
    class_weight='balanced_subsample'
)
lgb_base = LGBMClassifier(
    objective='multiclass', random_state=42, n_jobs=1, verbose=-1,
    subsample_freq=1, subsample=0.8, reg_lambda=0,
    num_leaves=22, n_estimators=700, min_child_samples=5,
    max_depth=-1, learning_rate=0.05,
    colsample_bytree=0.6, class_weight=None
)

# 클래스 혼동을 줄이기 위해 두 클래스 자료만으로 학습하는 보조 모델
pair_et = ExtraTreesClassifier(
    n_estimators=450, min_samples_leaf=2, max_features=0.9,
    random_state=42, n_jobs=1, class_weight='balanced'
)

# 2027 검증 분할에서 미리 선택한 한 가지 후보를 고정
SPECIALIST_CONFIDENCE = 0.58
MAX_PAIR_PROB_GAP = 0.30
MIN_PAIR_PROB_SUM = 0.50


def macro(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro', zero_division=0)


def evaluate(seed):
    outer = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    rows = []
    y_all, base_all, revised_all = [], [], []

    for fold, (tr, va) in enumerate(outer.split(X_dev, y_dev), 1):
        Xa, Xv = X_dev.iloc[tr], X_dev.iloc[va]
        ya, yv = y_dev[tr], y_dev[va]

        # 외부 학습 Fold에서만 OOF 확률 생성(스태킹 누수 방지)
        et_oof = cross_val_predict(
            clone(et_base), Xa, ya, cv=inner,
            method='predict_proba', n_jobs=1
        )
        lgb_oof = cross_val_predict(
            clone(lgb_base), Xa, ya, cv=inner,
            method='predict_proba', n_jobs=1
        )
        meta = LogisticRegression(max_iter=3000, C=1.0).fit(
            np.hstack([et_oof, lgb_oof]), ya
        )
        et_valid = clone(et_base).fit(Xa, ya).predict_proba(Xv)
        lgb_valid = clone(lgb_base).fit(Xa, ya).predict_proba(Xv)
        meta_valid = np.hstack([et_valid, lgb_valid])
        base_pred = meta.predict(meta_valid)
        base_prob = meta.predict_proba(meta_valid)

        # 외부 Fold의 학습 부분 중 Bumps / Other_Faults 사례만 사용
        mask_train = np.isin(ya, [B, O])
        pair_target = (ya[mask_train] == B).astype(int)
        specialist = clone(pair_et).fit(Xa.loc[mask_train], pair_target)
        prob_bumps = specialist.predict_proba(Xv)[:, 1]

        # 메인 모델이 Bumps 또는 Other_Faults로 분류했고,
        # 두 클래스에 대한 상대적 불확실성이 존재하는 경우만 후보로 지정
        eligible = (
            np.isin(base_pred, [B, O])
            & ((base_prob[:, B] + base_prob[:, O]) >= MIN_PAIR_PROB_SUM)
            & (np.abs(base_prob[:, B] - base_prob[:, O]) <= MAX_PAIR_PROB_GAP)
            & ((prob_bumps >= SPECIALIST_CONFIDENCE)
               | (prob_bumps <= 1 - SPECIALIST_CONFIDENCE))
        )
        revised = base_pred.copy()
        revised[eligible] = np.where(prob_bumps[eligible] >= 0.5, B, O)

        changed = (revised != base_pred)
        corrected = int(np.sum(changed & (revised == yv) & (base_pred != yv)))
        worsened = int(np.sum(changed & (base_pred == yv) & (revised != yv)))

        rows.append({
            'Fold': fold, '기존 스태킹': macro(yv, base_pred),
            '보조 분류기': macro(yv, revised),
            '변경 건수': int(changed.sum()),
            '정답으로 개선': corrected, '정답에서 오답': worsened
        })
        y_all.append(np.asarray(yv))
        base_all.append(base_pred)
        revised_all.append(revised)
        print(
            f'분할 {seed} Fold {fold} | 기존={rows[-1]["기존 스태킹"]:.4f}'
            f' | 개선후={rows[-1]["보조 분류기"]:.4f}'
            f' | 변경={int(changed.sum())}', flush=True
        )
    result = pd.DataFrame(rows)
    return result, np.concatenate(y_all), np.concatenate(base_all), np.concatenate(revised_all)

print('보조 분류기 설정 준비 완료')

보조 분류기 설정 준비 완료


In [3]:
# 6단계: Pastry / Other_Faults 전용 보조모델 추가
P = int(encoder.transform(['Pastry'])[0])
pair_pastry = ExtraTreesClassifier(
    n_estimators=450, min_samples_leaf=2, max_features=0.9,
    random_state=42, n_jobs=1, class_weight='balanced'
)

# 설정은 새로운 검증 분할에서 점수를 확인하기 전에 고정
BO_CONF, BO_GAP, BO_SUM = 0.58, 0.30, 0.50
PO_CONF, PO_GAP, PO_SUM = 0.65, 0.15, 0.50


def pair_revise(pred, probability, specialized_probability, left, right,
                confidence, max_gap, min_sum):
    """한 결함 쌍에 대해 높은 확신 & 쌍 내부 불확실성 조건을 만족할 때만 변경."""
    revised = pred.copy()
    eligible = (
        np.isin(pred, [left, right])
        & (probability[:, left] + probability[:, right] >= min_sum)
        & (np.abs(probability[:, left] - probability[:, right]) <= max_gap)
        & ((specialized_probability >= confidence)
           | (specialized_probability <= 1.0 - confidence))
    )
    revised[eligible] = np.where(specialized_probability[eligible] >= 0.5, left, right)
    return revised


def evaluate_stage6(seed=2053):
    outer = StratifiedKFold(n_splits=3, shuffle=True, random_state=seed)
    inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=1)
    rows = []
    collected = {name: [] for name in ['y', '기존 스태킹', 'B/O 보조', 'P/O 보조', 'B/O + P/O 보조']}
    for fold, (tr, va) in enumerate(outer.split(X_dev, y_dev), 1):
        Xa, Xv = X_dev.iloc[tr], X_dev.iloc[va]
        ya, yv = y_dev[tr], y_dev[va]

        # 원래 스태킹을 동일한 외부 Fold 내에서 재학습
        et_oof = cross_val_predict(clone(et_base), Xa, ya, cv=inner,
                                   method='predict_proba', n_jobs=1)
        lgb_oof = cross_val_predict(clone(lgb_base), Xa, ya, cv=inner,
                                    method='predict_proba', n_jobs=1)
        meta = LogisticRegression(max_iter=3000, C=1.0).fit(
            np.hstack([et_oof, lgb_oof]), ya
        )
        et_va = clone(et_base).fit(Xa, ya).predict_proba(Xv)
        lgb_va = clone(lgb_base).fit(Xa, ya).predict_proba(Xv)
        meta_input = np.hstack([et_va, lgb_va])
        base_pred = meta.predict(meta_input)
        probs = meta.predict_proba(meta_input)

        # 1번 보조 모델: Bumps / Other_Faults
        bo_train = np.isin(ya, [B, O])
        bo_binary = (ya[bo_train] == B).astype(int)
        prob_b = clone(pair_et).fit(
            Xa.iloc[np.flatnonzero(bo_train)], bo_binary
        ).predict_proba(Xv)[:, 1]
        revised_bo = pair_revise(base_pred, probs, prob_b, B, O,
                                 BO_CONF, BO_GAP, BO_SUM)

        # 2번 보조 모델: Pastry / Other_Faults
        po_train = np.isin(ya, [P, O])
        po_binary = (ya[po_train] == P).astype(int)
        prob_p = clone(pair_pastry).fit(
            Xa.iloc[np.flatnonzero(po_train)], po_binary
        ).predict_proba(Xv)[:, 1]
        revised_po = pair_revise(base_pred, probs, prob_p, P, O,
                                 PO_CONF, PO_GAP, PO_SUM)

        # 순차 적용: 먼저 B/O 판정한 결과가 P/O에 속하는 경우에만 P/O 사용
        # (B/O에서 Bumps로 변경된 샘플은 P/O 단계에서 다시 바꾸지 않음)
        revised_both = pair_revise(revised_bo, probs, prob_p, P, O,
                                   PO_CONF, PO_GAP, PO_SUM)

        preds = {
            '기존 스태킹': base_pred, 'B/O 보조': revised_bo,
            'P/O 보조': revised_po, 'B/O + P/O 보조': revised_both
        }
        r = {'Fold': fold}
        for label, pr in preds.items():
            r[label] = macro(yv, pr)
            collected[label].append(pr)
        r['B/O 변경'] = int((base_pred != revised_bo).sum())
        r['P/O 추가 변경'] = int((revised_bo != revised_both).sum())
        r['최종 변경'] = int((base_pred != revised_both).sum())
        r['정답 개선'] = int(((base_pred != yv) & (revised_both == yv)).sum())
        r['정답 악화'] = int(((base_pred == yv) & (revised_both != yv)).sum())
        rows.append(r)
        collected['y'].append(yv)
        print(f'외부 Fold {fold} 완료 | 기존={r["기존 스태킹"]:.4f} | '
              f'B/O={r["B/O 보조"]:.4f} | '
              f'P/O={r["P/O 보조"]:.4f} | '
              f'둘다={r["B/O + P/O 보조"]:.4f}', flush=True)

    joined = {k: np.concatenate(v) for k, v in collected.items()}
    return pd.DataFrame(rows), joined

print('6단계 평가 함수 준비 완료')

6단계 평가 함수 준비 완료


In [4]:
# [6-1] 설정을 고정한 다음, 기존과 다른 새로운 3-Fold로 평가
stage6_folds, stage6_predictions = evaluate_stage6(seed=2053)
print('\n[같은 외부 Fold별 모델 비교]')
display(stage6_folds.round(4))
model_cols = ['기존 스태킹', 'B/O 보조', 'P/O 보조', 'B/O + P/O 보조']
summary6 = pd.DataFrame({
    '평균 Macro F1': stage6_folds[model_cols].mean(),
    '표준편차': stage6_folds[model_cols].std(ddof=0),
    '기존 대비': stage6_folds[model_cols].mean() - stage6_folds['기존 스태킹'].mean(),
}).sort_values('평균 Macro F1', ascending=False)
print('\n[모델별 평균 성능 - 높은 순]')
display(summary6.round(4))
print('\nFold 평균 목표 0.835 달성 후보:',
      list(summary6.index[summary6['평균 Macro F1'] >= 0.835]))
print('주의: 후보 선택에 새 분할 결과를 사용했다면 성능을 확정하지 말고 검증을 추가하세요.')
stage6_folds.to_csv('stage6_fold_scores.csv', index=False, encoding='utf-8-sig')

외부 Fold 1 완료 | 기존=0.8026 | B/O=0.8043 | P/O=0.8033 | 둘다=0.8050
외부 Fold 2 완료 | 기존=0.7864 | B/O=0.7864 | P/O=0.7906 | 둘다=0.7906
외부 Fold 3 완료 | 기존=0.8195 | B/O=0.8205 | P/O=0.8132 | 둘다=0.8143

[같은 외부 Fold별 모델 비교]


,Fold,기존 스태킹,B/O 보조,P/O 보조,B/O + P/O 보조,B/O 변경,P/O 추가 변경,최종 변경,정답 개선,정답 악화
0,1,0.8026,0.8043,0.8033,0.8050,9,2,11,6,4
1,2,0.7864,0.7864,0.7906,0.7906,8,2,10,5,4
2,3,0.8195,0.8205,0.8132,0.8143,6,3,9,3,5



[모델별 평균 성능 - 높은 순]


,평균 Macro F1,표준편차,기존 대비
B/O 보조,0.8037,0.0139,0.0009
B/O + P/O 보조,0.8033,0.0098,0.0005
기존 스태킹,0.8028,0.0135,0.0000
P/O 보조,0.8024,0.0093,-0.0004



Fold 평균 목표 0.835 달성 후보: []
주의: 후보 선택에 새 분할 결과를 사용했다면 성능을 확정하지 말고 검증을 추가하세요.


In [5]:
# [6-2] 클래스별 F1 및 변경된 예측을 확인
true6 = stage6_predictions['y']
base6 = stage6_predictions['기존 스태킹']
bo6 = stage6_predictions['B/O 보조']
both6 = stage6_predictions['B/O + P/O 보조']
class_names = list(encoder.classes_)
class_f1 = pd.DataFrame({
    label: f1_score(true6, stage6_predictions[label],
                    average=None, labels=np.arange(len(class_names)), zero_division=0)
    for label in ['기존 스태킹', 'B/O 보조', 'P/O 보조', 'B/O + P/O 보조']
}, index=class_names)
class_f1['두 보조 모델 변화'] = class_f1['B/O + P/O 보조'] - class_f1['기존 스태킹']
print('[클래스별 F1]')
display(class_f1.round(4))
print('\n[두 보조 모델 적용 후 오분류 행렬]')
display(pd.DataFrame(confusion_matrix(true6, both6), index=class_names, columns=class_names))
print('\n[변경 건수 / 정답 개선 / 정답 악화]')
display(stage6_folds[['Fold','B/O 변경','P/O 추가 변경','최종 변경','정답 개선','정답 악화']])
print('\n이 결과는 개발 데이터 외부 교차검증입니다. 독립 테스트셋 389행은 아직 사용하지 않았습니다.')

[클래스별 F1]


,기존 스태킹,B/O 보조,P/O 보조,B/O + P/O 보조,두 보조 모델 변화
Bumps,0.6507,0.6562,0.6507,0.6562,0.0054
Dirtiness,0.8108,0.8108,0.8108,0.8108,0.0000
K_Scatch,0.9677,0.9677,0.9677,0.9677,0.0000
Other_Faults,0.7354,0.7364,0.7343,0.7353,-0.0001
Pastry,0.6063,0.6063,0.6036,0.6036,-0.0027
Stains,0.9217,0.9217,0.9217,0.9217,0.0000
Z_Scratch,0.9338,0.9338,0.9338,0.9338,0.0000



[두 보조 모델 적용 후 오분류 행렬]


,Bumps,Dirtiness,K_Scatch,Other_Faults,Pastry,Stains,Z_Scratch
Bumps,208,0,0,102,7,1,3
Dirtiness,6,30,0,7,1,0,0
K_Scatch,0,0,300,11,0,2,0
Other_Faults,88,0,6,418,21,1,4
Pastry,11,0,0,46,67,0,2
Stains,0,0,0,5,0,53,0
Z_Scratch,0,0,1,10,0,0,141



[변경 건수 / 정답 개선 / 정답 악화]


,Fold,B/O 변경,P/O 추가 변경,최종 변경,정답 개선,정답 악화
0,1,9,2,11,6,4
1,2,8,2,10,5,4
2,3,6,3,9,3,5



이 결과는 개발 데이터 외부 교차검증입니다. 독립 테스트셋 389행은 아직 사용하지 않았습니다.
